# Drift Detection con Evidently

**Objetivo:** Detectar si la distribución de features de logs ha cambiado respecto al período de entrenamiento.

**Hipótesis clave:** En producción, la infraestructura cambia (nuevos servidores, updates de software). Un modelo entrenado en datos históricos puede degradarse sin alertas. Evidently detecta ese drift automáticamente.

**Tipos de drift analizados:**
- **Data drift:** cambio en distribución de features (error_rate, burst_flag, severity_mean)
- **Drift simulado:** tomar el 20% final del dataset como "producción" y primeras semanas como referencia

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.features.engineering import load_features
from src.monitoring.drift_detector import DriftDetector

plt.rcParams['figure.dpi'] = 120

FEATURES_PATH  = Path('../data/processed/features_train.parquet')
REPORTS_PATH   = Path('../reports/figures')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

## 1. Cargar features y simular escenario de producción

In [ ]:
feat_df = load_features(FEATURES_PATH)
feature_cols = [c for c in feat_df.columns if c not in {'timestamp', 'node', 'is_anomaly'}]

n = len(feat_df)
split_80 = int(n * 0.8)
split_60 = int(n * 0.6)

reference_df = feat_df.iloc[:split_60][feature_cols].fillna(0)
production_real = feat_df.iloc[split_80:][feature_cols].fillna(0)  # últimas 2 semanas

# Simular drift artificial: shift en distribuciones
rng = np.random.default_rng(42)
production_drifted = production_real.copy()
for col in production_drifted.columns[:5]:  # driftar las primeras 5 features
    production_drifted[col] = production_drifted[col] + rng.normal(2, 0.5, len(production_drifted))

print(f'Referencia: {len(reference_df):,} muestras (primeras 60%)')
print(f'Producción real: {len(production_real):,} muestras (últimas 20%)')
print(f'Producción drifteada: {len(production_drifted):,} muestras (drift artificial)')
print(f'Features: {len(feature_cols)}')

## 2. Escenario A — Sin drift (producción real vs referencia)

In [ ]:
detector_a = DriftDetector(threshold=0.05)
detector_a.fit_reference(reference_df)
report_a = detector_a.detect(production_real)

print(f'Drift score: {report_a.drift_score:.4f}')
print(f'Drift detectado: {report_a.drift_detected}')
print(f'Features drifteadas: {report_a.features_drifted}')
print(f'Recomendación: {report_a.recommendation}')

## 3. Escenario B — Con drift artificial

In [ ]:
detector_b = DriftDetector(threshold=0.05)
detector_b.fit_reference(reference_df)
report_b = detector_b.generate_html_report_from_data(
    production_drifted,
    path=REPORTS_PATH / '08_drift_report.html'
)

print(f'Drift score: {report_b.drift_score:.4f}')
print(f'Drift detectado: {report_b.drift_detected}')
print(f'Features drifteadas ({len(report_b.features_drifted)}): {report_b.features_drifted[:5]}')
print(f'Recomendación: {report_b.recommendation}')
print()
print('Reporte HTML guardado en: reports/figures/08_drift_report.html')

## 4. Visualizar p-values por feature (KS test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, report, title, color in [
    (axes[0], report_a, 'Escenario A — Sin drift', '#4361ee'),
    (axes[1], report_b, 'Escenario B — Con drift artificial', '#ef233c'),
]:
    scores = report.feature_scores
    if not scores:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax.transAxes)
        continue

    features = list(scores.keys())[:15]  # top 15 para legibilidad
    pvals = [scores[f] for f in features]
    colors = ['#ef233c' if p < 0.05 else color for p in pvals]

    ax.barh(features, pvals, color=colors, alpha=0.8)
    ax.axvline(x=0.05, color='gray', linestyle='--', alpha=0.7, label='umbral p=0.05')
    ax.set_xlabel('p-value (KS test)')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig(REPORTS_PATH / '08_drift_pvalues.png', dpi=150)
plt.show()

## 5. Comparativa — Distribución antes y después del drift

In [ ]:
# Visualizar las features más afectadas
top_drifted = sorted(report_b.feature_scores.items(), key=lambda x: x[1])[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (col, pval) in zip(axes.flat, top_drifted):
    if col not in reference_df.columns:
        continue
    ax.hist(reference_df[col].dropna(), bins=40, alpha=0.6, label='Referencia', color='#4361ee', density=True)
    ax.hist(production_drifted[col].dropna(), bins=40, alpha=0.6, label='Producción (drifted)', color='#ef233c', density=True)
    ax.set_title(f'{col}\np-value: {pval:.2e}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Distribuciones — Features con mayor drift', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_PATH / '08_drift_distributions.png', dpi=150)
plt.show()

## 6. Resumen y decisión de re-entrenamiento

**¿Cuándo re-entrenar?**

| Drift score | Acción recomendada |
|---|---|
| 0.0 – 0.15 | Sin acción — modelo estable |
| 0.15 – 0.30 | Monitorear — revisar en 3 días |
| 0.30 – 0.70 | Considerar re-entrenamiento |
| > 0.70 | Re-entrenamiento urgente con `/retrain` |

El umbral de 0.30 está justificado empiricamente: por debajo, los cambios en features son ruido estadístico. Por encima, el modelo empieza a ver distribuciones significativamente distintas a las de entrenamiento y las métricas de producción se degradan.

In [ ]:
print('=== Resumen Drift Detection ===')
print(f'Escenario A (producción real):      drift_score={report_a.drift_score:.4f}  → {report_a.recommendation}')
print(f'Escenario B (drift artificial):     drift_score={report_b.drift_score:.4f}  → {report_b.recommendation}')
print()
print('Endpoint disponible:')
print('  GET /model/health → drift_score, drift_detected, features_drifted, recommendation')